# 🎲 Exercícios — Raciocínio Probabilístico e Redes Bayesianas

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Pratique o Teorema de Bayes, Naive Bayes e redes bayesianas simples.


## 1. Teorema de Bayes — Diagnóstico Médico

In [ ]:
# Exemplo: diagnóstico de doença rara
# P(doença) = 0.001 (1 em 1000 pessoas tem a doença)
# P(teste+|doença) = 0.99 (sensibilidade: 99%)
# P(teste+|saudável) = 0.05 (taxa de falso positivo: 5%)

p_doenca = 0.001
p_saudavel = 1 - p_doenca
p_positivo_dado_doente = 0.99
p_positivo_dado_saudavel = 0.05

# P(teste+) = P(+|D)*P(D) + P(+|S)*P(S)
p_positivo = p_positivo_dado_doente*p_doenca + p_positivo_dado_saudavel*p_saudavel

# P(doença|teste+) = P(+|D)*P(D) / P(+)
p_doenca_dado_positivo = (p_positivo_dado_doente * p_doenca) / p_positivo

print("=== Análise Bayesiana do Teste Médico ===")
print(f"P(doença)              = {p_doenca:.4f} ({p_doenca*100:.1f}%)")
print(f"P(positivo)            = {p_positivo:.4f} ({p_positivo*100:.1f}%)")
print(f"P(doença | positivo)   = {p_doenca_dado_positivo:.4f} ({p_doenca_dado_positivo*100:.2f}%)")
print()
print("⚠️  Surpreendente: mesmo com teste 99% preciso,")
print(f"    a chance real de ter a doença após teste positivo é {p_doenca_dado_positivo*100:.1f}%!")
print("    Isso se deve à baixa prevalência da doença (base rate neglect).")

# Visualização
import matplotlib.pyplot as plt
import numpy as np

prevalencias = np.linspace(0.001, 0.5, 200)
prob_pos_cond = (0.99*prevalencias) / (0.99*prevalencias + 0.05*(1-prevalencias))
plt.figure(figsize=(8,4))
plt.plot(prevalencias*100, prob_pos_cond*100, 'b-', linewidth=2)
plt.axvline(p_doenca*100, color='red', linestyle='--', label=f'Prevalência atual ({p_doenca*100:.1f}%)')
plt.xlabel('Prevalência da Doença (%)'); plt.ylabel('P(doença|teste+) (%)')
plt.title('P(doença|positivo) vs Prevalência'); plt.legend(); plt.grid(True); plt.show()


### 📝 Exercício 1

Calcule a probabilidade de estar **saudável** mesmo após dois testes positivos consecutivos (assumindo testes independentes). Compare com a probabilidade após apenas 1 teste positivo.

In [ ]:
# ✏️ Calcule P(doença | dois testes positivos)
# Dica: use o resultado do primeiro teste como nova prior

# Após 1 teste positivo: P(D|+1) = p_doenca_dado_positivo
prior_2 = p_doenca_dado_positivo  # nova prior

# P(D|+1, +2) = Bayes novamente
p_doenca_2_positivos = (p_positivo_dado_doente * prior_2) /     (p_positivo_dado_doente*prior_2 + p_positivo_dado_saudavel*(1-prior_2))

print(f"Após 1 teste positivo: P(doença) = {p_doenca_dado_positivo:.4f} ({p_doenca_dado_positivo*100:.2f}%)")
print(f"Após 2 testes positivos: P(doença) = {p_doenca_2_positivos:.4f} ({p_doenca_2_positivos*100:.2f}%)")


## 2. Naive Bayes — Classificador de Spam

In [ ]:
from collections import defaultdict
import re, math

# Dataset de treinamento (simplificado)
emails = [
    ("ganhe dinheiro rapido oferta exclusiva gratis", "spam"),
    ("oferta imperdivel ganhe premio agora click", "spam"),
    ("gratis para sempre compre agora promoção", "spam"),
    ("reunião amanha pauta agenda conferencia", "ham"),
    ("relatório projeto prazo entrega semana", "ham"),
    ("aniversario colega confraternização festa", "ham"),
    ("oferta especial ganhe premio clique aqui gratis", "spam"),
    ("planilha dados analise resultados trimestre", "ham"),
    ("dinheiro facil ganhe sem trabalhar click", "spam"),
    ("proposta projeto cliente reuniao segunda", "ham"),
]

class NaiveBayesSpam:
    def __init__(self):
        self.contagens = defaultdict(lambda: defaultdict(int))
        self.totais = defaultdict(int)
        self.n_docs = defaultdict(int)
    
    def treinar(self, dados):
        for texto, classe in dados:
            palavras = re.findall(r'\w+', texto.lower())
            self.n_docs[classe] += 1
            for p in palavras:
                self.contagens[classe][p] += 1
                self.totais[classe] += 1
    
    def log_verossimilhanca(self, palavra, classe, vocab_size):
        """Suavização de Laplace."""
        return math.log((self.contagens[classe][palavra]+1) /
                        (self.totais[classe]+vocab_size))
    
    def classificar(self, texto):
        palavras = re.findall(r'\w+', texto.lower())
        vocab = set(p for cls in self.contagens.values() for p in cls)
        V = len(vocab)
        n_total = sum(self.n_docs.values())
        scores = {}
        for classe in self.n_docs:
            log_prior = math.log(self.n_docs[classe]/n_total)
            log_liks = sum(self.log_verossimilhanca(p,classe,V) for p in palavras)
            scores[classe] = log_prior + log_liks
        return max(scores, key=scores.get), scores

nb = NaiveBayesSpam()
nb.treinar(emails)

testes = [
    "oferta exclusiva ganhe dinheiro",
    "reunião de equipe amanha",
    "clique gratis premio exclusivo",
    "projeto resultados analise dados",
]
print(f"{'Email':<40} | {'Classe':^6} | {'Confiança'}")
print("-"*70)
for email in testes:
    classe, scores = nb.classificar(email)
    dif = scores['spam'] - scores['ham']
    print(f"{email:<40} | {classe:^6} | {'Alta' if abs(dif)>2 else 'Média'}")


### 📝 Exercício 2

Adicione **5 novos emails** ao dataset de treinamento e reclassifique os emails de teste. A acurácia melhora?

In [ ]:
emails_extras = [
    # ✏️ Adicione 5 emails (texto, "spam" ou "ham")
    ("produto milagroso emagrecimento rapido compre ja", "spam"),
    ("ata reuniao semana passada arquivo anexo", "ham"),
    # ...
]

nb2 = NaiveBayesSpam()
nb2.treinar(emails + emails_extras)

for email in testes:
    classe, _ = nb2.classificar(email)
    print(f"{email[:40]:<40} → {classe}")


## 3. Rede Bayesiana Simples

In [ ]:
# Rede Bayesiana: Chuva → Aspersor → Grama Molhada ← Chuva
# (Exemplo clássico de Pearl)

# Tabelas de probabilidade condicional (CPT)
P_chuva = {True: 0.2, False: 0.8}

P_aspersor_dado_chuva = {
    True:  {True: 0.01, False: 0.99},
    False: {True: 0.40, False: 0.60},
}

P_molhada_dado_chuva_aspersor = {
    (True,  True):  {True: 0.99, False: 0.01},
    (True,  False): {True: 0.80, False: 0.20},
    (False, True):  {True: 0.90, False: 0.10},
    (False, False): {True: 0.00, False: 1.00},
}

def inferir_por_enumeracao(evidencias):
    """Calcula P(chuva | evidências) por enumeração."""
    result = {}
    for chuva in [True, False]:
        prob = P_chuva[chuva]
        for aspersor in [True, False]:
            p_asp = P_aspersor_dado_chuva[chuva][aspersor]
            for molhada in [True, False]:
                p_mol = P_molhada_dado_chuva_aspersor[(chuva,aspersor)][molhada]
                p_total = prob * p_asp * p_mol
                # verifica evidências
                ev_ok = all(
                    (chuva if k=='chuva' else aspersor if k=='aspersor' else molhada) == v
                    for k,v in evidencias.items()
                )
                if ev_ok:
                    result[chuva] = result.get(chuva,0) + p_total
    total = sum(result.values())
    return {k: v/total for k, v in result.items()} if total > 0 else {}

# Consultas
print("Distribuição marginal P(chuva):", P_chuva)
print()
print("P(chuva | grama_molhada=True):")
r = inferir_por_enumeracao({"molhada": True})
print(f"  P(chuva=True  | molhada) = {r.get(True,0):.4f}")
print(f"  P(chuva=False | molhada) = {r.get(False,0):.4f}")
print()
print("P(chuva | grama_molhada=True, aspersor=False):")
r2 = inferir_por_enumeracao({"molhada": True, "aspersor": False})
print(f"  P(chuva=True  | molhada, ¬aspersor) = {r2.get(True,0):.4f}")
print(f"  P(chuva=False | molhada, ¬aspersor) = {r2.get(False,0):.4f}")


### 📝 Exercício Final

Adicionando um nó **"previsão_chuva"** que influencia P(chuva), calcule:
- P(chuva | previsão=True, grama_molhada=True)
- P(chuva | previsão=False, grama_molhada=True)

O que muda com a evidência adicional da previsão?

In [ ]:
# ✏️ Estenda a rede bayesiana com o nó 'previsao_chuva':
P_previsao = {True: 0.4, False: 0.6}
P_chuva_dado_previsao = {
    True:  {True: 0.6, False: 0.4},   # previsão certa na maioria
    False: {True: 0.1, False: 0.9},
}

# TODO: adapte inferir_por_enumeracao para incluir o nó de previsão
# e compute as consultas acima
